In [ ]:
#@title Cell 28.1 - Notebook overview
# This cell states the purpose, model boundary and expected outputs of Notebook 28.

from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 28: Model C BioSample-Level MIC Prediction Matrix

## Purpose

Notebook 28 will use the validated Model C reference model to complete the
BioSample--antibiotic MIC matrix.

It will:

1. load the final Model C model and embeddings from Notebook 26;
2. load the nearest-training-pathogen similarity results from Notebook 27;
3. use the independent nested pathogen-out residuals to calculate one
   antibiotic-specific 95% prediction interval;
4. predict every pathogen--antibiotic combination in

\[
9{,}058\times26=235{,}508
\]

matrix cells;
5. preserve the 50,460 observed MIC values without replacement;
6. fill only the 185,048 genuine blank cells with Model C predictions;
7. attach lower and upper 95% prediction limits to every predicted MIC;
8. attach the nearest-training-pathogen similarity and prediction-support
   status to every matrix cell;
9. identify predictions outside the observed MIC range for each antibiotic;
10. construct numerical, status and readable MIC matrices; and
11. save the completed Model C matrix in CSV, Excel and ZIP formats.

## Model boundary

Notebook 28 will not change the Model C pathogen kernel, the selected sequence
kernel, \(\rho=0.4\), the 256 pathogen coordinates, the 26 antibiotic
coordinates or the Ridge penalty \(\alpha=1\).

The 95% prediction intervals and prediction-support information are post-model
calculations. They do not alter the central MIC predictions.

This notebook will not compare Model 3, Model 3B and Model C. That comparison
will be performed separately in Notebook 29.

Clinical breakpoint interpretation and treatment recommendations are outside
the scope of this notebook.

## Expected notebook length

Notebook 28 contains **12 cells**.
"""))

print(
    "Transition: Cell 28.2 will import the required packages, mount "
    "Google Drive and define the Notebook 28 settings."
)


In [ ]:
#@title Cell 28.2 - Import packages and define notebook settings
# This cell imports the required tools, mounts Google Drive and defines all fixed paths and dimensions.

import gc
import hashlib
import json
import os
import shutil
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

from google.colab import drive

drive.mount("/content/drive")

EXPECTED_MODEL_C_PATHOGENS = 9_058
EXPECTED_ANTIBIOTICS = 26
EXPECTED_OBSERVED_INTERACTIONS = 50_460
EXPECTED_TOTAL_MATRIX_CELLS = (
    EXPECTED_MODEL_C_PATHOGENS
    * EXPECTED_ANTIBIOTICS
)
EXPECTED_PREDICTED_BLANK_CELLS = (
    EXPECTED_TOTAL_MATRIX_CELLS
    - EXPECTED_OBSERVED_INTERACTIONS
)

EXPECTED_SEQUENCE_KERNEL = "locus-weighted"
EXPECTED_RHO = 0.4
EXPECTED_PATHOGEN_DIMENSION = 256
EXPECTED_ANTIBIOTIC_DIMENSION = 26
EXPECTED_RIDGE_ALPHA = 1.0
EXPECTED_INTERACTION_PREDICTORS = (
    EXPECTED_PATHOGEN_DIMENSION
    * EXPECTED_ANTIBIOTIC_DIMENSION
)

PREDICTION_PATHOGEN_BLOCK_SIZE = 48

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/Model3_MIC_Project"
)

NOTEBOOK28_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook28"
)

NOTEBOOK28_RESULT_DIRECTORY = (
    NOTEBOOK28_DIRECTORY
    / "results"
)

WORK_DIRECTORY = Path(
    "/content/notebook28_work"
)

NOTEBOOK26_INPUT_DIRECTORY = (
    WORK_DIRECTORY
    / "notebook26_inputs"
)

NOTEBOOK27_INPUT_DIRECTORY = (
    WORK_DIRECTORY
    / "notebook27_inputs"
)

for directory in [
    NOTEBOOK28_DIRECTORY,
    NOTEBOOK28_RESULT_DIRECTORY,
    WORK_DIRECTORY,
    NOTEBOOK26_INPUT_DIRECTORY,
    NOTEBOOK27_INPUT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

NOTEBOOK26_ARCHIVE_NAME = (
    "26_model_c_nested_pathogen_out_and_reference_model_outputs.zip"
)

NOTEBOOK27_ARCHIVE_NAME = (
    "27_model_c_prediction_support_outputs.zip"
)


def file_sha256(file_path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(file_path, "rb") as input_file:
        while True:
            block = input_file.read(block_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


settings_summary = pd.DataFrame(
    [
        {
            "setting": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "setting": "Antibiotics",
            "value": EXPECTED_ANTIBIOTICS,
        },
        {
            "setting": "Observed MIC cells",
            "value": EXPECTED_OBSERVED_INTERACTIONS,
        },
        {
            "setting": "Blank cells to predict",
            "value": EXPECTED_PREDICTED_BLANK_CELLS,
        },
        {
            "setting": "Pathogen coordinates",
            "value": EXPECTED_PATHOGEN_DIMENSION,
        },
        {
            "setting": "Interaction predictors",
            "value": EXPECTED_INTERACTION_PREDICTORS,
        },
        {
            "setting": "Notebook 28 output directory",
            "value": str(NOTEBOOK28_RESULT_DIRECTORY),
        },
    ]
)

display(settings_summary)

print(
    "Notebook 28 packages, directories and fixed settings were "
    "defined successfully."
)

print(
    "\nTransition: Cell 28.3 will locate, validate and extract "
    "the required Notebook 26 and Notebook 27 archives."
)


In [ ]:
#@title Cell 28.3 - Locate and extract the required input archives
# This cell finds the Notebook 26 and 27 ZIP archives, validates their contents and extracts the required files locally.


def locate_unique_archive(archive_name):
    candidates = sorted(
        PROJECT_DIRECTORY.rglob(archive_name)
    )

    if not candidates:
        raise FileNotFoundError(
            f"{archive_name} was not found under "
            f"{PROJECT_DIRECTORY}. Place the archive anywhere "
            "inside Model3_MIC_Project and rerun this cell."
        )

    if len(candidates) == 1:
        return candidates[0]

    archive_hashes = {
        file_sha256(path)
        for path in candidates
    }

    if len(archive_hashes) != 1:
        raise ValueError(
            f"Multiple different copies of {archive_name} were found: "
            f"{candidates}"
        )

    print(
        f"Multiple identical copies of {archive_name} were found. "
        f"Using: {candidates[0]}"
    )

    return candidates[0]


def archive_member_by_basename(archive, required_name):
    matches = [
        member
        for member in archive.namelist()
        if Path(member).name == required_name
    ]

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one archive member named {required_name}; "
            f"observed {matches}."
        )

    return matches[0]


def extract_and_validate_archive(
    archive_path,
    destination_directory,
    manifest_name,
    required_names,
):
    destination_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(archive_path, "r") as archive:
        damaged_member = archive.testzip()

        if damaged_member is not None:
            raise ValueError(
                f"Damaged archive member: {damaged_member}"
            )

        names_to_extract = list(
            dict.fromkeys(
                [manifest_name, *required_names]
            )
        )

        extracted_paths = {}

        for required_name in names_to_extract:
            member = archive_member_by_basename(
                archive,
                required_name,
            )

            output_path = (
                destination_directory
                / required_name
            )

            with archive.open(member) as source_file:
                with open(output_path, "wb") as output_file:
                    shutil.copyfileobj(
                        source_file,
                        output_file,
                    )

            extracted_paths[required_name] = output_path

    with open(
        extracted_paths[manifest_name],
        "r",
        encoding="utf-8",
    ) as manifest_file:
        manifest = json.load(manifest_file)

    if manifest.get("validation_status") != "passed":
        raise ValueError(
            f"{manifest_name} does not report passed validation."
        )

    manifest_file_records = {
        record["file_name"]: record
        for record in manifest.get("files", [])
    }

    for required_name in required_names:
        if required_name not in manifest_file_records:
            raise ValueError(
                f"{required_name} is not listed in {manifest_name}."
            )

        observed_hash = file_sha256(
            extracted_paths[required_name]
        )

        expected_hash = manifest_file_records[
            required_name
        ]["sha256"]

        if observed_hash != expected_hash:
            raise ValueError(
                f"Checksum mismatch for {required_name}."
            )

    return manifest, extracted_paths


notebook26_archive_path = locate_unique_archive(
    NOTEBOOK26_ARCHIVE_NAME
)

notebook27_archive_path = locate_unique_archive(
    NOTEBOOK27_ARCHIVE_NAME
)

NOTEBOOK26_REQUIRED_FILES = [
    "26_model_c_aligned_interactions.csv.gz",
    "26_nested_pathogen_out_predictions.csv.gz",
    "26_final_model_c_reference_model.joblib",
    "26_final_model_c_embeddings.npz",
    "26_final_model_c_pathogen_embedding_index.csv",
    "26_final_model_c_antibiotic_embedding_index.csv",
    "26_final_model_c_configuration.json",
]

NOTEBOOK27_REQUIRED_FILES = [
    "27_pathogen_nearest_training_similarity.csv",
    "27_model_c_prediction_support_configuration.json",
]

(
    notebook26_manifest,
    notebook26_files,
) = extract_and_validate_archive(
    notebook26_archive_path,
    NOTEBOOK26_INPUT_DIRECTORY,
    "26_output_manifest.json",
    NOTEBOOK26_REQUIRED_FILES,
)

(
    notebook27_manifest,
    notebook27_files,
) = extract_and_validate_archive(
    notebook27_archive_path,
    NOTEBOOK27_INPUT_DIRECTORY,
    "27_output_manifest.json",
    NOTEBOOK27_REQUIRED_FILES,
)

input_summary = pd.DataFrame(
    [
        {
            "input": "Notebook 26 archive",
            "path": str(notebook26_archive_path),
            "validation_status": notebook26_manifest[
                "validation_status"
            ],
        },
        {
            "input": "Notebook 27 archive",
            "path": str(notebook27_archive_path),
            "validation_status": notebook27_manifest[
                "validation_status"
            ],
        },
    ]
)

display(input_summary)

print(
    "Both input archives and all required files passed checksum "
    "validation."
)

print(
    "\nTransition: Cell 28.4 will load and validate the final Model C "
    "model, embeddings, observed MIC table and prediction-support table."
)


In [ ]:
#@title Cell 28.4 - Load and validate the Model C inputs
# This cell loads the final model, embeddings, MIC interactions, nested predictions and pathogen-support information.


def require_columns(table, required_columns, table_name):
    missing_columns = [
        column
        for column in required_columns
        if column not in table.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{table_name} is missing columns {missing_columns}. "
            f"Observed columns: {list(table.columns)}"
        )


with open(
    notebook26_files[
        "26_final_model_c_configuration.json"
    ],
    "r",
    encoding="utf-8",
) as input_file:
    model_c_configuration = json.load(input_file)

with open(
    notebook27_files[
        "27_model_c_prediction_support_configuration.json"
    ],
    "r",
    encoding="utf-8",
) as input_file:
    prediction_support_configuration = json.load(input_file)

final_model_c = joblib.load(
    notebook26_files[
        "26_final_model_c_reference_model.joblib"
    ]
)

embedding_archive = np.load(
    notebook26_files[
        "26_final_model_c_embeddings.npz"
    ]
)

required_embedding_arrays = {
    "pathogen_embedding",
    "pathogen_eigenvalues",
    "antibiotic_embedding",
}

missing_embedding_arrays = (
    required_embedding_arrays
    - set(embedding_archive.files)
)

if missing_embedding_arrays:
    raise ValueError(
        "The final embedding archive is missing arrays: "
        f"{sorted(missing_embedding_arrays)}"
    )

pathogen_embedding = np.asarray(
    embedding_archive["pathogen_embedding"],
    dtype=np.float32,
)

pathogen_eigenvalues = np.asarray(
    embedding_archive["pathogen_eigenvalues"],
    dtype=np.float64,
)

antibiotic_embedding = np.asarray(
    embedding_archive["antibiotic_embedding"],
    dtype=np.float32,
)

pathogen_embedding_index = pd.read_csv(
    notebook26_files[
        "26_final_model_c_pathogen_embedding_index.csv"
    ]
)

antibiotic_embedding_index = pd.read_csv(
    notebook26_files[
        "26_final_model_c_antibiotic_embedding_index.csv"
    ]
)

model_c_interactions = pd.read_csv(
    notebook26_files[
        "26_model_c_aligned_interactions.csv.gz"
    ]
)

nested_predictions = pd.read_csv(
    notebook26_files[
        "26_nested_pathogen_out_predictions.csv.gz"
    ]
)

pathogen_support = pd.read_csv(
    notebook27_files[
        "27_pathogen_nearest_training_similarity.csv"
    ]
)

require_columns(
    pathogen_embedding_index,
    [
        "pathogen_embedding_row",
        "model_c_row_index",
        "biosample",
        "assembly_accession",
    ],
    "Pathogen embedding index",
)

require_columns(
    antibiotic_embedding_index,
    [
        "antibiotic_embedding_row",
        "antibiotic",
    ],
    "Antibiotic embedding index",
)

require_columns(
    model_c_interactions,
    [
        "biosample",
        "antibiotic",
        "log2_mic",
        "model_c_pathogen_row",
        "antibiotic_coordinate_row",
    ],
    "Model C interaction table",
)

require_columns(
    nested_predictions,
    [
        "biosample",
        "antibiotic",
        "observed_log2_mic",
        "predicted_log2_mic",
        "model",
        "outer_fold",
    ],
    "Nested prediction table",
)

require_columns(
    pathogen_support,
    [
        "model_c_row_index",
        "biosample",
        "assembly_accession",
        "nearest_training_similarity",
    ],
    "Pathogen-support table",
)

pathogen_embedding_index = (
    pathogen_embedding_index
    .sort_values("pathogen_embedding_row")
    .reset_index(drop=True)
)

antibiotic_embedding_index = (
    antibiotic_embedding_index
    .sort_values("antibiotic_embedding_row")
    .reset_index(drop=True)
)

if not np.array_equal(
    pathogen_embedding_index[
        "pathogen_embedding_row"
    ].to_numpy(),
    np.arange(EXPECTED_MODEL_C_PATHOGENS),
):
    raise ValueError(
        "Pathogen embedding rows are not the complete ordered range 0 to 9,057."
    )

if not np.array_equal(
    antibiotic_embedding_index[
        "antibiotic_embedding_row"
    ].to_numpy(),
    np.arange(EXPECTED_ANTIBIOTICS),
):
    raise ValueError(
        "Antibiotic embedding rows are not the complete ordered range 0 to 25."
    )

if pathogen_embedding.shape != (
    EXPECTED_MODEL_C_PATHOGENS,
    EXPECTED_PATHOGEN_DIMENSION,
):
    raise ValueError(
        f"Unexpected pathogen embedding shape: {pathogen_embedding.shape}"
    )

if antibiotic_embedding.shape != (
    EXPECTED_ANTIBIOTICS,
    EXPECTED_ANTIBIOTIC_DIMENSION,
):
    raise ValueError(
        f"Unexpected antibiotic embedding shape: {antibiotic_embedding.shape}"
    )

if len(model_c_interactions) != EXPECTED_OBSERVED_INTERACTIONS:
    raise ValueError(
        f"Expected {EXPECTED_OBSERVED_INTERACTIONS:,} observed interactions; "
        f"observed {len(model_c_interactions):,}."
    )

if model_c_interactions.duplicated(
    ["biosample", "antibiotic"]
).any():
    raise ValueError(
        "The observed interaction table contains duplicate BioSample--antibiotic pairs."
    )

model_c_nested_predictions = (
    nested_predictions[
        nested_predictions["model"]
        == "Model C"
    ]
    .copy()
)

if len(model_c_nested_predictions) != EXPECTED_OBSERVED_INTERACTIONS:
    raise ValueError(
        "The nested prediction table does not contain exactly one independent "
        "Model C prediction for every observed MIC."
    )

if model_c_nested_predictions.duplicated(
    ["biosample", "antibiotic"]
).any():
    raise ValueError(
        "The nested Model C predictions contain duplicate BioSample--antibiotic pairs."
    )

if len(pathogen_support) != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError(
        f"Expected {EXPECTED_MODEL_C_PATHOGENS:,} pathogen-support rows; "
        f"observed {len(pathogen_support):,}."
    )

if pathogen_support["model_c_row_index"].duplicated().any():
    raise ValueError(
        "The pathogen-support table contains duplicate Model C pathogen rows."
    )

ordered_support = (
    pathogen_embedding_index[
        [
            "model_c_row_index",
            "biosample",
            "assembly_accession",
        ]
    ]
    .merge(
        pathogen_support[
            [
                "model_c_row_index",
                "biosample",
                "assembly_accession",
                "nearest_training_similarity",
            ]
        ],
        on=[
            "model_c_row_index",
            "biosample",
            "assembly_accession",
        ],
        how="left",
        validate="one_to_one",
    )
)

if ordered_support[
    "nearest_training_similarity"
].isna().any():
    raise ValueError(
        "The pathogen order does not match the Notebook 27 support table."
    )

configuration_checks = {
    "sequence_kernel": (
        model_c_configuration["selected_sequence_kernel"]
        == EXPECTED_SEQUENCE_KERNEL
    ),
    "rho": np.isclose(
        model_c_configuration["selected_rho"],
        EXPECTED_RHO,
    ),
    "pathogen_dimension": (
        model_c_configuration["selected_pathogen_dimension"]
        == EXPECTED_PATHOGEN_DIMENSION
    ),
    "ridge_alpha": np.isclose(
        model_c_configuration["selected_ridge_alpha"],
        EXPECTED_RIDGE_ALPHA,
    ),
    "interaction_predictors": (
        model_c_configuration["interaction_predictors"]
        == EXPECTED_INTERACTION_PREDICTORS
    ),
    "support_validation": (
        prediction_support_configuration["validation_status"]
        == "passed"
    ),
}

if not all(configuration_checks.values()):
    failed_checks = [
        name
        for name, passed in configuration_checks.items()
        if not passed
    ]

    raise ValueError(
        f"Model C configuration validation failed: {failed_checks}"
    )

EMPIRICAL_SUPPORT_BOUNDARY = float(
    prediction_support_configuration[
        "empirical_support_boundary"
    ]
)

input_validation_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": pathogen_embedding.shape[0],
        },
        {
            "metric": "Observed MIC interactions",
            "value": len(model_c_interactions),
        },
        {
            "metric": "Independent nested predictions",
            "value": len(model_c_nested_predictions),
        },
        {
            "metric": "Antibiotics",
            "value": antibiotic_embedding.shape[0],
        },
        {
            "metric": "Pathogen coordinates",
            "value": pathogen_embedding.shape[1],
        },
        {
            "metric": "Ridge alpha",
            "value": model_c_configuration[
                "selected_ridge_alpha"
            ],
        },
        {
            "metric": "Empirical support boundary",
            "value": EMPIRICAL_SUPPORT_BOUNDARY,
        },
        {
            "metric": "Input validation status",
            "value": "passed",
        },
    ]
)

display(input_validation_summary)

print(
    "All Model C model, MIC and prediction-support inputs were loaded "
    "and validated."
)

print(
    "\nTransition: Cell 28.5 will calculate antibiotic-specific 95% "
    "prediction intervals from the independent nested pathogen-out residuals."
)


In [ ]:
#@title Cell 28.5 - Calculate antibiotic-specific 95% prediction intervals
# This cell uses independent nested pathogen-out residuals to define lower and upper prediction limits for each antibiotic.

model_c_nested_predictions[
    "residual_log2_mic"
] = (
    model_c_nested_predictions[
        "observed_log2_mic"
    ]
    - model_c_nested_predictions[
        "predicted_log2_mic"
    ]
)

if not np.isfinite(
    model_c_nested_predictions[
        [
            "observed_log2_mic",
            "predicted_log2_mic",
            "residual_log2_mic",
        ]
    ].to_numpy()
).all():
    raise ValueError(
        "The nested prediction residual table contains invalid numerical values."
    )

interval_rows = []

for antibiotic in antibiotic_embedding_index[
    "antibiotic"
]:
    antibiotic_predictions = (
        model_c_nested_predictions[
            model_c_nested_predictions[
                "antibiotic"
            ]
            == antibiotic
        ]
        .copy()
    )

    if antibiotic_predictions.empty:
        raise ValueError(
            f"No independent nested predictions were found for {antibiotic}."
        )

    residuals = antibiotic_predictions[
        "residual_log2_mic"
    ].to_numpy(dtype=np.float64)

    lower_residual = float(
        np.quantile(
            residuals,
            0.025,
            method="linear",
        )
    )

    upper_residual = float(
        np.quantile(
            residuals,
            0.975,
            method="linear",
        )
    )

    lower_limits = (
        antibiotic_predictions[
            "predicted_log2_mic"
        ].to_numpy(dtype=np.float64)
        + lower_residual
    )

    upper_limits = (
        antibiotic_predictions[
            "predicted_log2_mic"
        ].to_numpy(dtype=np.float64)
        + upper_residual
    )

    observed_values = antibiotic_predictions[
        "observed_log2_mic"
    ].to_numpy(dtype=np.float64)

    empirical_coverage = float(
        np.mean(
            (observed_values >= lower_limits)
            & (observed_values <= upper_limits)
        )
    )

    interval_rows.append(
        {
            "antibiotic": antibiotic,
            "independent_residuals": len(residuals),
            "lower_residual_2_5_percentile": lower_residual,
            "upper_residual_97_5_percentile": upper_residual,
            "prediction_interval_width_log2_mic": (
                upper_residual
                - lower_residual
            ),
            "empirical_residual_coverage": empirical_coverage,
        }
    )

prediction_interval_calibration = pd.DataFrame(
    interval_rows
)

if len(prediction_interval_calibration) != EXPECTED_ANTIBIOTICS:
    raise ValueError(
        "The interval calibration table does not contain all 26 antibiotics."
    )

if (
    prediction_interval_calibration[
        "lower_residual_2_5_percentile"
    ]
    > prediction_interval_calibration[
        "upper_residual_97_5_percentile"
    ]
).any():
    raise ValueError(
        "At least one antibiotic has reversed prediction limits."
    )

PREDICTION_INTERVAL_CALIBRATION_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_antibiotic_prediction_interval_calibration.csv"
)

prediction_interval_calibration.to_csv(
    PREDICTION_INTERVAL_CALIBRATION_PATH,
    index=False,
)

display(
    prediction_interval_calibration.round(4)
)

print(
    "The antibiotic-specific 95% prediction intervals were calculated "
    "from independent nested pathogen-out residuals."
)

print(
    f"Saved: {PREDICTION_INTERVAL_CALIBRATION_PATH}"
)

print(
    "\nTransition: Cell 28.6 will predict the complete 9,058 by 26 "
    "BioSample--antibiotic grid in restartable pathogen blocks."
)


In [ ]:
#@title Cell 28.6 - Predict the complete Model C MIC grid
# This cell predicts all 235,508 matrix cells in memory-safe, restartable pathogen blocks.

FULL_GRID_PREDICTION_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_full_grid_predictions.npy"
)

FULL_GRID_PREDICTION_METADATA_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_full_grid_prediction_checkpoint.json"
)

prediction_checkpoint_signature = {
    "model_sha256": file_sha256(
        notebook26_files[
            "26_final_model_c_reference_model.joblib"
        ]
    ),
    "embedding_sha256": file_sha256(
        notebook26_files[
            "26_final_model_c_embeddings.npz"
        ]
    ),
    "pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "antibiotics": EXPECTED_ANTIBIOTICS,
    "pathogen_dimension": EXPECTED_PATHOGEN_DIMENSION,
    "antibiotic_dimension": EXPECTED_ANTIBIOTIC_DIMENSION,
}

expected_prediction_shape = (
    EXPECTED_MODEL_C_PATHOGENS,
    EXPECTED_ANTIBIOTICS,
)

if FULL_GRID_PREDICTION_PATH.exists():
    if not FULL_GRID_PREDICTION_METADATA_PATH.exists():
        raise FileNotFoundError(
            "The saved prediction checkpoint has no metadata. Rename or "
            "remove the checkpoint before restarting this cell."
        )

    with open(
        FULL_GRID_PREDICTION_METADATA_PATH,
        "r",
        encoding="utf-8",
    ) as metadata_file:
        saved_prediction_signature = json.load(
            metadata_file
        )

    if saved_prediction_signature != prediction_checkpoint_signature:
        raise ValueError(
            "The saved prediction checkpoint was produced from different "
            "Model C inputs. Rename or remove it before restarting this cell."
        )

    grid_prediction_matrix = np.load(
        FULL_GRID_PREDICTION_PATH,
        mmap_mode="r+",
    )

    if grid_prediction_matrix.shape != expected_prediction_shape:
        raise ValueError(
            "The saved prediction checkpoint has the wrong dimensions. "
            "Rename or remove it before restarting this cell."
        )

    print(
        "Existing prediction checkpoint detected. Missing pathogen rows "
        "will be recalculated."
    )

else:
    grid_prediction_matrix = np.lib.format.open_memmap(
        FULL_GRID_PREDICTION_PATH,
        mode="w+",
        dtype=np.float32,
        shape=expected_prediction_shape,
    )

    grid_prediction_matrix[:] = np.nan
    grid_prediction_matrix.flush()

    with open(
        FULL_GRID_PREDICTION_METADATA_PATH,
        "w",
        encoding="utf-8",
    ) as metadata_file:
        json.dump(
            prediction_checkpoint_signature,
            metadata_file,
            indent=2,
        )

    print(
        "A new restartable prediction checkpoint was created."
    )

processed_blocks = 0

for pathogen_start in range(
    0,
    EXPECTED_MODEL_C_PATHOGENS,
    PREDICTION_PATHOGEN_BLOCK_SIZE,
):
    pathogen_stop = min(
        pathogen_start
        + PREDICTION_PATHOGEN_BLOCK_SIZE,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    saved_block = np.asarray(
        grid_prediction_matrix[
            pathogen_start:pathogen_stop,
            :,
        ]
    )

    if np.isfinite(saved_block).all():
        continue

    block_pathogen_embedding = pathogen_embedding[
        pathogen_start:pathogen_stop,
        :,
    ]

    block_pathogen_rows = np.repeat(
        block_pathogen_embedding,
        EXPECTED_ANTIBIOTICS,
        axis=0,
    )

    block_antibiotic_rows = np.tile(
        antibiotic_embedding,
        (
            pathogen_stop - pathogen_start,
            1,
        ),
    )

    block_interaction_features = np.einsum(
        "ij,ik->ijk",
        block_pathogen_rows,
        block_antibiotic_rows,
    ).reshape(
        -1,
        EXPECTED_INTERACTION_PREDICTORS,
    ).astype(
        np.float32,
        copy=False,
    )

    block_predictions = final_model_c.predict(
        block_interaction_features
    ).reshape(
        pathogen_stop - pathogen_start,
        EXPECTED_ANTIBIOTICS,
    )

    if not np.isfinite(block_predictions).all():
        raise ValueError(
            f"Invalid predictions were produced for pathogen rows "
            f"{pathogen_start} to {pathogen_stop - 1}."
        )

    grid_prediction_matrix[
        pathogen_start:pathogen_stop,
        :,
    ] = block_predictions.astype(np.float32)

    grid_prediction_matrix.flush()

    processed_blocks += 1

    if (
        processed_blocks % 10 == 0
        or pathogen_stop == EXPECTED_MODEL_C_PATHOGENS
    ):
        print(
            f"Predicted pathogen rows 0 to {pathogen_stop - 1:,} "
            f"of {EXPECTED_MODEL_C_PATHOGENS - 1:,}."
        )

    del (
        block_pathogen_rows,
        block_antibiotic_rows,
        block_interaction_features,
        block_predictions,
    )

    gc.collect()

if not np.isfinite(
    np.asarray(grid_prediction_matrix)
).all():
    raise ValueError(
        "The full prediction checkpoint still contains missing or invalid values."
    )

print(
    "\nComplete Model C interaction grid predicted and checkpointed."
)

print(
    f"Grid dimensions: {EXPECTED_MODEL_C_PATHOGENS:,} × "
    f"{EXPECTED_ANTIBIOTICS}"
)

print(
    f"Total cells: {EXPECTED_TOTAL_MATRIX_CELLS:,}"
)

print(
    "\nTransition: Cell 28.7 will preserve observed MICs, fill genuine "
    "blank cells and attach prediction limits and support information."
)


In [ ]:
#@title Cell 28.7 - Combine observed and predicted MIC values
# This cell preserves observed MICs, fills blank cells and attaches 95% prediction limits and prediction-support information.

biosample_order = pathogen_embedding_index[
    "biosample"
].astype(str).to_numpy()

assembly_order = pathogen_embedding_index[
    "assembly_accession"
].astype(str).to_numpy()

model_c_row_order = pathogen_embedding_index[
    "model_c_row_index"
].astype(int).to_numpy()

antibiotic_order = antibiotic_embedding_index[
    "antibiotic"
].astype(str).to_numpy()

antibiotic_row_order = antibiotic_embedding_index[
    "antibiotic_embedding_row"
].astype(int).to_numpy()

complete_prediction_grid = pd.DataFrame(
    {
        "model_c_row_index": np.repeat(
            model_c_row_order,
            EXPECTED_ANTIBIOTICS,
        ),
        "biosample": np.repeat(
            biosample_order,
            EXPECTED_ANTIBIOTICS,
        ),
        "assembly_accession": np.repeat(
            assembly_order,
            EXPECTED_ANTIBIOTICS,
        ),
        "antibiotic_embedding_row": np.tile(
            antibiotic_row_order,
            EXPECTED_MODEL_C_PATHOGENS,
        ),
        "antibiotic": np.tile(
            antibiotic_order,
            EXPECTED_MODEL_C_PATHOGENS,
        ),
        "model_c_predicted_log2_mic": np.asarray(
            grid_prediction_matrix,
            dtype=np.float32,
        ).reshape(-1),
    }
)

observed_interactions = (
    model_c_interactions[
        [
            "biosample",
            "antibiotic",
            "log2_mic",
        ]
    ]
    .rename(
        columns={
            "log2_mic": "observed_log2_mic"
        }
    )
)

final_mic_panel_long = (
    complete_prediction_grid
    .merge(
        observed_interactions,
        on=["biosample", "antibiotic"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        prediction_interval_calibration,
        on="antibiotic",
        how="left",
        validate="many_to_one",
    )
    .merge(
        ordered_support[
            [
                "model_c_row_index",
                "biosample",
                "assembly_accession",
                "nearest_training_similarity",
            ]
        ],
        on=[
            "model_c_row_index",
            "biosample",
            "assembly_accession",
        ],
        how="left",
        validate="many_to_one",
    )
)

final_mic_panel_long[
    "cell_status"
] = np.where(
    final_mic_panel_long[
        "observed_log2_mic"
    ].notna(),
    "Observed",
    "Predicted",
)

final_mic_panel_long[
    "final_log2_mic"
] = final_mic_panel_long[
    "observed_log2_mic"
].fillna(
    final_mic_panel_long[
        "model_c_predicted_log2_mic"
    ]
)

final_mic_panel_long[
    "model_c_predicted_mic_mg_l"
] = np.exp2(
    final_mic_panel_long[
        "model_c_predicted_log2_mic"
    ]
)

final_mic_panel_long[
    "final_mic_mg_l"
] = np.exp2(
    final_mic_panel_long[
        "final_log2_mic"
    ]
)

predicted_cell_mask = (
    final_mic_panel_long[
        "cell_status"
    ]
    == "Predicted"
)

final_mic_panel_long[
    "lower_95_prediction_limit_log2_mic"
] = np.where(
    predicted_cell_mask,
    final_mic_panel_long[
        "model_c_predicted_log2_mic"
    ]
    + final_mic_panel_long[
        "lower_residual_2_5_percentile"
    ],
    np.nan,
)

final_mic_panel_long[
    "upper_95_prediction_limit_log2_mic"
] = np.where(
    predicted_cell_mask,
    final_mic_panel_long[
        "model_c_predicted_log2_mic"
    ]
    + final_mic_panel_long[
        "upper_residual_97_5_percentile"
    ],
    np.nan,
)

final_mic_panel_long[
    "lower_95_prediction_limit_mic_mg_l"
] = np.where(
    predicted_cell_mask,
    np.exp2(
        final_mic_panel_long[
            "lower_95_prediction_limit_log2_mic"
        ]
    ),
    np.nan,
)

final_mic_panel_long[
    "upper_95_prediction_limit_mic_mg_l"
] = np.where(
    predicted_cell_mask,
    np.exp2(
        final_mic_panel_long[
            "upper_95_prediction_limit_log2_mic"
        ]
    ),
    np.nan,
)

final_mic_panel_long[
    "empirical_support_boundary"
] = EMPIRICAL_SUPPORT_BOUNDARY

final_mic_panel_long[
    "evaluated_similarity_status"
] = np.where(
    final_mic_panel_long[
        "nearest_training_similarity"
    ]
    >= EMPIRICAL_SUPPORT_BOUNDARY,
    "inside evaluated similarity range",
    "outside evaluated similarity range",
)

final_mic_panel_long[
    "prediction_support_status"
] = np.where(
    final_mic_panel_long[
        "cell_status"
    ]
    == "Observed",
    "observed MIC; prediction support not required",
    final_mic_panel_long[
        "evaluated_similarity_status"
    ],
)

biosample_observation_counts = (
    observed_interactions
    .groupby("biosample")
    .size()
)

antibiotic_observation_counts = (
    observed_interactions
    .groupby("antibiotic")
    .size()
)

final_mic_panel_long[
    "biosample_observed_antibiotics"
] = (
    final_mic_panel_long["biosample"]
    .map(biosample_observation_counts)
    .fillna(0)
    .astype(int)
)

final_mic_panel_long[
    "antibiotic_training_interactions"
] = (
    final_mic_panel_long["antibiotic"]
    .map(antibiotic_observation_counts)
    .fillna(0)
    .astype(int)
)

antibiotic_observed_ranges = (
    observed_interactions
    .groupby("antibiotic")[
        "observed_log2_mic"
    ]
    .agg(
        observed_minimum_log2_mic="min",
        observed_maximum_log2_mic="max",
    )
    .reset_index()
)

final_mic_panel_long = (
    final_mic_panel_long
    .merge(
        antibiotic_observed_ranges,
        on="antibiotic",
        how="left",
        validate="many_to_one",
    )
)

final_mic_panel_long[
    "outside_observed_antibiotic_range"
] = (
    predicted_cell_mask
    & (
        (
            final_mic_panel_long[
                "model_c_predicted_log2_mic"
            ]
            < final_mic_panel_long[
                "observed_minimum_log2_mic"
            ]
        )
        | (
            final_mic_panel_long[
                "model_c_predicted_log2_mic"
            ]
            > final_mic_panel_long[
                "observed_maximum_log2_mic"
            ]
        )
    )
)

print(
    "Observed and predicted MIC values, 95% prediction limits and "
    "prediction-support information were combined."
)

display(
    final_mic_panel_long[
        "cell_status"
    ].value_counts().rename_axis(
        "cell_status"
    ).reset_index(name="matrix_cells")
)

print(
    "\nTransition: Cell 28.8 will construct the numerical, status and "
    "readable BioSample-level MIC matrices."
)


In [ ]:
#@title Cell 28.8 - Construct the final Model C MIC matrices
# This cell converts the long table into ordered wide matrices for central MICs, prediction limits and support information.


def ordered_matrix(value_column):
    return (
        final_mic_panel_long
        .pivot(
            index="biosample",
            columns="antibiotic",
            values=value_column,
        )
        .reindex(
            index=biosample_order,
            columns=antibiotic_order,
        )
    )


final_log2_mic_matrix = ordered_matrix(
    "final_log2_mic"
)

final_mic_mg_l_matrix = ordered_matrix(
    "final_mic_mg_l"
)

lower_95_prediction_limit_log2_matrix = ordered_matrix(
    "lower_95_prediction_limit_log2_mic"
)

upper_95_prediction_limit_log2_matrix = ordered_matrix(
    "upper_95_prediction_limit_log2_mic"
)

lower_95_prediction_limit_mg_l_matrix = ordered_matrix(
    "lower_95_prediction_limit_mic_mg_l"
)

upper_95_prediction_limit_mg_l_matrix = ordered_matrix(
    "upper_95_prediction_limit_mic_mg_l"
)

final_cell_status_matrix = ordered_matrix(
    "cell_status"
)

prediction_support_status_matrix = ordered_matrix(
    "prediction_support_status"
)

outside_observed_range_matrix = ordered_matrix(
    "outside_observed_antibiotic_range"
)

readable_mic_panel = pd.DataFrame(
    index=biosample_order,
    columns=antibiotic_order,
    dtype="object",
)

for antibiotic in antibiotic_order:
    readable_values = []

    for biosample in biosample_order:
        status = final_cell_status_matrix.loc[
            biosample,
            antibiotic,
        ]

        final_mic = final_mic_mg_l_matrix.loc[
            biosample,
            antibiotic,
        ]

        if status == "Observed":
            readable_values.append(
                f"O: {final_mic:.3g}"
            )
            continue

        lower_limit = (
            lower_95_prediction_limit_mg_l_matrix.loc[
                biosample,
                antibiotic,
            ]
        )

        upper_limit = (
            upper_95_prediction_limit_mg_l_matrix.loc[
                biosample,
                antibiotic,
            ]
        )

        support_status = (
            prediction_support_status_matrix.loc[
                biosample,
                antibiotic,
            ]
        )

        outside_range = bool(
            outside_observed_range_matrix.loc[
                biosample,
                antibiotic,
            ]
        )

        flags = ""

        if outside_range:
            flags += "!"

        if support_status == "outside evaluated similarity range":
            flags += "†"

        readable_values.append(
            f"P{flags}: {final_mic:.3g} "
            f"[{lower_limit:.3g}, {upper_limit:.3g}]"
        )

    readable_mic_panel[antibiotic] = readable_values

readable_mic_panel.index.name = "biosample"

preferred_preview_biosample = "SAMN16304089"

if preferred_preview_biosample in readable_mic_panel.index:
    preview_biosample = preferred_preview_biosample
else:
    preview_biosample = biosample_observation_counts.idxmax()

print("The ordered Model C MIC matrices were constructed.")
print("\nReadable-panel notation:")
print("O = observed MIC in mg/L")
print("P = Model C predicted MIC in mg/L [lower, upper 95% prediction limits]")
print("! = central prediction outside that antibiotic's observed MIC range")
print("† = pathogen outside the evaluated similarity range")

print(
    f"\nBioSample preview: {preview_biosample}"
)

display(
    readable_mic_panel.loc[
        [preview_biosample]
    ]
)

print(
    "\nTransition: Cell 28.9 will validate every matrix, observed MIC "
    "preservation, prediction limit and prediction-support label."
)


In [ ]:
#@title Cell 28.9 - Validate the completed Model C MIC matrices
# This cell confirms all dimensions, counts, numerical values, observed values, prediction limits and support labels.

expected_matrix_shape = (
    EXPECTED_MODEL_C_PATHOGENS,
    EXPECTED_ANTIBIOTICS,
)

matrices_to_validate = {
    "final_log2_mic_matrix": final_log2_mic_matrix,
    "final_mic_mg_l_matrix": final_mic_mg_l_matrix,
    "lower_95_prediction_limit_log2_matrix": (
        lower_95_prediction_limit_log2_matrix
    ),
    "upper_95_prediction_limit_log2_matrix": (
        upper_95_prediction_limit_log2_matrix
    ),
    "lower_95_prediction_limit_mg_l_matrix": (
        lower_95_prediction_limit_mg_l_matrix
    ),
    "upper_95_prediction_limit_mg_l_matrix": (
        upper_95_prediction_limit_mg_l_matrix
    ),
    "final_cell_status_matrix": final_cell_status_matrix,
    "prediction_support_status_matrix": (
        prediction_support_status_matrix
    ),
}

wrong_shapes = {
    name: matrix.shape
    for name, matrix in matrices_to_validate.items()
    if matrix.shape != expected_matrix_shape
}

if wrong_shapes:
    raise ValueError(
        f"Final matrix dimensions are incorrect: {wrong_shapes}"
    )

status_counts = final_mic_panel_long[
    "cell_status"
].value_counts()

observed_count = int(
    status_counts.get("Observed", 0)
)

predicted_count = int(
    status_counts.get("Predicted", 0)
)

unresolved_count = int(
    EXPECTED_TOTAL_MATRIX_CELLS
    - observed_count
    - predicted_count
)

observed_rows = final_mic_panel_long[
    final_mic_panel_long["cell_status"]
    == "Observed"
]

predicted_rows = final_mic_panel_long[
    final_mic_panel_long["cell_status"]
    == "Predicted"
]

observed_values_preserved = np.allclose(
    observed_rows["final_log2_mic"],
    observed_rows["observed_log2_mic"],
)

prediction_limit_columns = [
    "lower_95_prediction_limit_log2_mic",
    "upper_95_prediction_limit_log2_mic",
    "lower_95_prediction_limit_mic_mg_l",
    "upper_95_prediction_limit_mic_mg_l",
]

predicted_limits_complete = np.isfinite(
    predicted_rows[
        prediction_limit_columns
    ].to_numpy(dtype=np.float64)
).all()

observed_limits_empty = observed_rows[
    prediction_limit_columns
].isna().all().all()

prediction_limits_ordered = (
    (
        predicted_rows[
            "lower_95_prediction_limit_log2_mic"
        ]
        <= predicted_rows[
            "upper_95_prediction_limit_log2_mic"
        ]
    ).all()
    and (
        predicted_rows[
            "lower_95_prediction_limit_mic_mg_l"
        ]
        <= predicted_rows[
            "upper_95_prediction_limit_mic_mg_l"
        ]
    ).all()
)

central_values_finite = np.isfinite(
    final_mic_panel_long[
        [
            "model_c_predicted_log2_mic",
            "final_log2_mic",
            "final_mic_mg_l",
        ]
    ].to_numpy(dtype=np.float64)
).all()

support_values_finite = np.isfinite(
    final_mic_panel_long[
        "nearest_training_similarity"
    ].to_numpy(dtype=np.float64)
).all()

valid_support_labels = set(
    final_mic_panel_long[
        "prediction_support_status"
    ].unique()
).issubset(
    {
        "observed MIC; prediction support not required",
        "inside evaluated similarity range",
        "outside evaluated similarity range",
    }
)

validation_checks = {
    "total_cells_correct": (
        len(final_mic_panel_long)
        == EXPECTED_TOTAL_MATRIX_CELLS
    ),
    "observed_cells_correct": (
        observed_count
        == EXPECTED_OBSERVED_INTERACTIONS
    ),
    "predicted_cells_correct": (
        predicted_count
        == EXPECTED_PREDICTED_BLANK_CELLS
    ),
    "unresolved_cells_zero": (
        unresolved_count == 0
    ),
    "observed_values_preserved": observed_values_preserved,
    "central_values_finite": central_values_finite,
    "predicted_limits_complete": predicted_limits_complete,
    "observed_limits_empty": observed_limits_empty,
    "prediction_limits_ordered": prediction_limits_ordered,
    "support_values_finite": support_values_finite,
    "support_labels_valid": valid_support_labels,
}

if not all(validation_checks.values()):
    failed_checks = [
        name
        for name, passed in validation_checks.items()
        if not passed
    ]

    raise ValueError(
        f"Notebook 28 validation failed: {failed_checks}"
    )

prediction_support_summary = (
    final_mic_panel_long
    .groupby(
        [
            "cell_status",
            "prediction_support_status",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "matrix_cells"})
)

final_validation_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Antibiotics",
            "value": EXPECTED_ANTIBIOTICS,
        },
        {
            "metric": "Total matrix cells",
            "value": EXPECTED_TOTAL_MATRIX_CELLS,
        },
        {
            "metric": "Observed MIC cells preserved",
            "value": observed_count,
        },
        {
            "metric": "Predicted blank cells",
            "value": predicted_count,
        },
        {
            "metric": "Unresolved cells",
            "value": unresolved_count,
        },
        {
            "metric": "Predicted cells with 95% prediction limits",
            "value": int(
                predicted_rows[
                    "lower_95_prediction_limit_log2_mic"
                ].notna().sum()
            ),
        },
        {
            "metric": "Predicted cells outside evaluated similarity range",
            "value": int(
                (
                    predicted_rows[
                        "prediction_support_status"
                    ]
                    == "outside evaluated similarity range"
                ).sum()
            ),
        },
        {
            "metric": "Predictions outside observed antibiotic MIC range",
            "value": int(
                predicted_rows[
                    "outside_observed_antibiotic_range"
                ].sum()
            ),
        },
        {
            "metric": "Empirical support boundary",
            "value": EMPIRICAL_SUPPORT_BOUNDARY,
        },
        {
            "metric": "Notebook 28 validation status",
            "value": "passed",
        },
    ]
)

MODEL_SUMMARY_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_production_model_summary.csv"
)

VALIDATION_SUMMARY_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_final_validation_summary.csv"
)

SUPPORT_SUMMARY_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_prediction_support_summary.csv"
)

model_summary = pd.DataFrame(
    [
        {
            "metric": "Model",
            "value": "Model C",
        },
        {
            "metric": "Sequence kernel",
            "value": EXPECTED_SEQUENCE_KERNEL,
        },
        {
            "metric": "Sequence contribution rho",
            "value": EXPECTED_RHO,
        },
        {
            "metric": "Pathogen coordinates",
            "value": EXPECTED_PATHOGEN_DIMENSION,
        },
        {
            "metric": "Antibiotic coordinates",
            "value": EXPECTED_ANTIBIOTIC_DIMENSION,
        },
        {
            "metric": "Interaction predictors",
            "value": EXPECTED_INTERACTION_PREDICTORS,
        },
        {
            "metric": "Ridge alpha",
            "value": EXPECTED_RIDGE_ALPHA,
        },
        {
            "metric": "Nested pathogen-out MAE",
            "value": notebook26_manifest[
                "nested_model_c_performance"
            ]["mae"],
        },
        {
            "metric": "Nested pathogen-out RMSE",
            "value": notebook26_manifest[
                "nested_model_c_performance"
            ]["rmse"],
        },
        {
            "metric": "Nested pathogen-out Pearson r",
            "value": notebook26_manifest[
                "nested_model_c_performance"
            ]["pearson_r"],
        },
    ]
)

model_summary.to_csv(
    MODEL_SUMMARY_PATH,
    index=False,
)

final_validation_summary.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False,
)

prediction_support_summary.to_csv(
    SUPPORT_SUMMARY_PATH,
    index=False,
)

display(final_validation_summary)

print("\nPrediction-support summary")
display(prediction_support_summary)

print(
    "\nAll Notebook 28 validation checks passed."
)

print(
    "\nTransition: Cell 28.10 will export the validated long table, "
    "wide matrices, model copy and configuration files."
)


In [ ]:
#@title Cell 28.10 - Export the validated Model C datasets
# This cell saves the long panel, all numerical and status matrices, support table, model copy and configuration.

LONG_PANEL_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_final_mic_panel_long.csv.gz"
)

FINAL_LOG2_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_final_log2_mic_matrix.csv"
)

FINAL_MIC_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_final_mic_mg_l_matrix.csv"
)

LOWER_LOG2_LIMIT_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_lower_95_prediction_limit_log2_mic_matrix.csv"
)

UPPER_LOG2_LIMIT_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_upper_95_prediction_limit_log2_mic_matrix.csv"
)

LOWER_MIC_LIMIT_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_lower_95_prediction_limit_mic_mg_l_matrix.csv"
)

UPPER_MIC_LIMIT_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_upper_95_prediction_limit_mic_mg_l_matrix.csv"
)

CELL_STATUS_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_final_cell_status_matrix.csv"
)

PREDICTION_SUPPORT_MATRIX_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_prediction_support_status_matrix.csv"
)

READABLE_PANEL_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_final_readable_mic_panel.csv"
)

PATHOGEN_SUPPORT_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_pathogen_prediction_support.csv"
)

MODEL_COPY_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_production_model.joblib"
)

CONFIGURATION_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_biosample_level_mic_prediction_configuration.json"
)

final_mic_panel_long.to_csv(
    LONG_PANEL_PATH,
    index=False,
    compression="gzip",
)

final_log2_mic_matrix.to_csv(
    FINAL_LOG2_MATRIX_PATH
)

final_mic_mg_l_matrix.to_csv(
    FINAL_MIC_MATRIX_PATH
)

lower_95_prediction_limit_log2_matrix.to_csv(
    LOWER_LOG2_LIMIT_MATRIX_PATH
)

upper_95_prediction_limit_log2_matrix.to_csv(
    UPPER_LOG2_LIMIT_MATRIX_PATH
)

lower_95_prediction_limit_mg_l_matrix.to_csv(
    LOWER_MIC_LIMIT_MATRIX_PATH
)

upper_95_prediction_limit_mg_l_matrix.to_csv(
    UPPER_MIC_LIMIT_MATRIX_PATH
)

final_cell_status_matrix.to_csv(
    CELL_STATUS_MATRIX_PATH
)

prediction_support_status_matrix.to_csv(
    PREDICTION_SUPPORT_MATRIX_PATH
)

readable_mic_panel.to_csv(
    READABLE_PANEL_PATH
)

ordered_support.assign(
    empirical_support_boundary=(
        EMPIRICAL_SUPPORT_BOUNDARY
    ),
    evaluated_similarity_status=np.where(
        ordered_support[
            "nearest_training_similarity"
        ]
        >= EMPIRICAL_SUPPORT_BOUNDARY,
        "inside evaluated similarity range",
        "outside evaluated similarity range",
    ),
).to_csv(
    PATHOGEN_SUPPORT_PATH,
    index=False,
)

shutil.copy2(
    notebook26_files[
        "26_final_model_c_reference_model.joblib"
    ],
    MODEL_COPY_PATH,
)

notebook28_configuration = {
    "notebook": 28,
    "model": "Model C",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "antibiotics": EXPECTED_ANTIBIOTICS,
    "total_matrix_cells": EXPECTED_TOTAL_MATRIX_CELLS,
    "observed_mic_cells": observed_count,
    "predicted_blank_cells": predicted_count,
    "unresolved_cells": unresolved_count,
    "selected_sequence_kernel": EXPECTED_SEQUENCE_KERNEL,
    "selected_rho": EXPECTED_RHO,
    "selected_pathogen_dimension": EXPECTED_PATHOGEN_DIMENSION,
    "antibiotic_coordinates": EXPECTED_ANTIBIOTIC_DIMENSION,
    "interaction_predictors": EXPECTED_INTERACTION_PREDICTORS,
    "selected_ridge_alpha": EXPECTED_RIDGE_ALPHA,
    "prediction_interval_method": (
        "For each antibiotic, add the 2.5th and 97.5th percentiles "
        "of independent nested pathogen-out residuals to the central "
        "Model C prediction."
    ),
    "prediction_intervals_change_model": False,
    "empirical_support_boundary": EMPIRICAL_SUPPORT_BOUNDARY,
    "support_definition": prediction_support_configuration[
        "boundary_definition"
    ],
    "clinical_breakpoint_interpretation": "not performed",
    "clinical_warning": (
        "Predicted MIC values and prediction limits are research outputs. "
        "They are not treatment recommendations and require confirmatory "
        "susceptibility testing."
    ),
    "validation_status": "passed",
}

with open(
    CONFIGURATION_PATH,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        notebook28_configuration,
        output_file,
        indent=2,
    )

exported_files = [
    LONG_PANEL_PATH,
    FINAL_LOG2_MATRIX_PATH,
    FINAL_MIC_MATRIX_PATH,
    LOWER_LOG2_LIMIT_MATRIX_PATH,
    UPPER_LOG2_LIMIT_MATRIX_PATH,
    LOWER_MIC_LIMIT_MATRIX_PATH,
    UPPER_MIC_LIMIT_MATRIX_PATH,
    CELL_STATUS_MATRIX_PATH,
    PREDICTION_SUPPORT_MATRIX_PATH,
    READABLE_PANEL_PATH,
    PATHOGEN_SUPPORT_PATH,
    PREDICTION_INTERVAL_CALIBRATION_PATH,
    MODEL_SUMMARY_PATH,
    VALIDATION_SUMMARY_PATH,
    SUPPORT_SUMMARY_PATH,
    MODEL_COPY_PATH,
    CONFIGURATION_PATH,
]

missing_exports = [
    path
    for path in exported_files
    if not path.exists()
]

if missing_exports:
    raise FileNotFoundError(
        f"The following Notebook 28 exports are missing: {missing_exports}"
    )

export_summary = pd.DataFrame(
    [
        {
            "file": path.name,
            "size_MB": round(
                path.stat().st_size / (1024 ** 2),
                3,
            ),
        }
        for path in exported_files
    ]
)

display(export_summary)

print(
    "All validated Notebook 28 CSV, model and configuration files were saved."
)

print(
    "\nTransition: Cell 28.11 will create and validate the colour-coded "
    "Excel workbook."
)


In [ ]:
#@title Cell 28.11 - Create the colour-coded Excel workbook
# This cell creates an Excel workbook containing central MICs, 95% prediction limits, cell status and support information.

EXCEL_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_model_c_biosample_level_mic_prediction_matrix.xlsx"
)

excel_legend = pd.DataFrame(
    [
        {
            "code": "O",
            "meaning": "Observed laboratory MIC in mg/L",
            "colour": "Green",
        },
        {
            "code": "P",
            "meaning": (
                "Model C predicted MIC in mg/L followed by lower and "
                "upper 95% prediction limits"
            ),
            "colour": "Blue",
        },
        {
            "code": "P!",
            "meaning": (
                "Central predicted MIC outside the observed MIC range "
                "for that antibiotic"
            ),
            "colour": "Orange",
        },
        {
            "code": "P†",
            "meaning": "Pathogen outside the evaluated similarity range",
            "colour": "Red",
        },
    ]
)

with pd.ExcelWriter(
    EXCEL_PATH,
    engine="openpyxl",
) as excel_writer:
    readable_mic_panel.to_excel(
        excel_writer,
        sheet_name="Readable_MIC_panel",
    )

    final_mic_mg_l_matrix.to_excel(
        excel_writer,
        sheet_name="Final_MIC_mg_L",
    )

    final_log2_mic_matrix.to_excel(
        excel_writer,
        sheet_name="Final_log2_MIC",
    )

    lower_95_prediction_limit_mg_l_matrix.to_excel(
        excel_writer,
        sheet_name="Lower_95_limit_mg_L",
    )

    upper_95_prediction_limit_mg_l_matrix.to_excel(
        excel_writer,
        sheet_name="Upper_95_limit_mg_L",
    )

    final_cell_status_matrix.to_excel(
        excel_writer,
        sheet_name="Cell_status",
    )

    prediction_support_status_matrix.to_excel(
        excel_writer,
        sheet_name="Prediction_support",
    )

    prediction_interval_calibration.to_excel(
        excel_writer,
        sheet_name="Interval_calibration",
        index=False,
    )

    final_validation_summary.to_excel(
        excel_writer,
        sheet_name="Validation_summary",
        index=False,
    )

    model_summary.to_excel(
        excel_writer,
        sheet_name="Model_summary",
        index=False,
    )

    excel_legend.to_excel(
        excel_writer,
        sheet_name="Legend",
        index=False,
    )

workbook = load_workbook(EXCEL_PATH)

panel_sheet = workbook["Readable_MIC_panel"]
panel_sheet.freeze_panes = "B2"
panel_sheet.auto_filter.ref = panel_sheet.dimensions

header_fill = PatternFill(
    fill_type="solid",
    fgColor="1F4E78",
)

header_font = Font(
    color="FFFFFF",
    bold=True,
)

observed_fill = PatternFill(
    fill_type="solid",
    fgColor="C6EFCE",
)

predicted_fill = PatternFill(
    fill_type="solid",
    fgColor="DDEBF7",
)

outside_range_fill = PatternFill(
    fill_type="solid",
    fgColor="FCE4D6",
)

outside_support_fill = PatternFill(
    fill_type="solid",
    fgColor="FFC7CE",
)

for cell in panel_sheet[1]:
    cell.fill = header_fill
    cell.font = header_font
    cell.alignment = Alignment(horizontal="center")

for row in panel_sheet.iter_rows(
    min_row=2,
    min_col=2,
):
    for cell in row:
        cell_value = str(cell.value)

        if cell_value.startswith("O:"):
            cell.fill = observed_fill
        elif "†" in cell_value:
            cell.fill = outside_support_fill
        elif cell_value.startswith("P!:"):
            cell.fill = outside_range_fill
        else:
            cell.fill = predicted_fill

        cell.alignment = Alignment(
            horizontal="center"
        )

panel_sheet.column_dimensions["A"].width = 20

for column_number in range(
    2,
    EXPECTED_ANTIBIOTICS + 2,
):
    column_letter = get_column_letter(
        column_number
    )
    panel_sheet.column_dimensions[
        column_letter
    ].width = 28

workbook.save(EXCEL_PATH)

with zipfile.ZipFile(
    EXCEL_PATH,
    "r",
) as excel_archive:
    damaged_member = excel_archive.testzip()

if damaged_member is not None:
    raise ValueError(
        f"The Excel workbook contains a damaged member: {damaged_member}"
    )

exported_files.append(EXCEL_PATH)

print(f"Saved and validated: {EXCEL_PATH}")

print(
    "\nTransition: Cell 28.12 will create the output manifest, package "
    "all validated files and report the final Notebook 28 status."
)


In [ ]:
#@title Cell 28.12 - Package and report the final Notebook 28 outputs
# This cell records checksums, creates one validated ZIP archive on Google Drive and reports the final matrix status.

OUTPUT_MANIFEST_PATH = (
    NOTEBOOK28_RESULT_DIRECTORY
    / "28_output_manifest.json"
)

FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK28_DIRECTORY
    / "28_model_c_biosample_level_mic_prediction_outputs.zip"
)

files_to_package = list(
    dict.fromkeys(exported_files)
)

missing_output_files = [
    path
    for path in files_to_package
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        f"Notebook 28 output files are missing: {missing_output_files}"
    )

output_manifest = {
    "notebook": 28,
    "model": "Model C",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "antibiotics": EXPECTED_ANTIBIOTICS,
    "matrix_shape": [
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_ANTIBIOTICS,
    ],
    "total_matrix_cells": EXPECTED_TOTAL_MATRIX_CELLS,
    "observed_mic_cells": observed_count,
    "predicted_blank_cells": predicted_count,
    "predicted_cells_with_95_percent_prediction_limits": int(
        predicted_rows[
            "lower_95_prediction_limit_log2_mic"
        ].notna().sum()
    ),
    "empirical_support_boundary": EMPIRICAL_SUPPORT_BOUNDARY,
    "selected_sequence_kernel": EXPECTED_SEQUENCE_KERNEL,
    "selected_rho": EXPECTED_RHO,
    "selected_pathogen_dimension": EXPECTED_PATHOGEN_DIMENSION,
    "selected_ridge_alpha": EXPECTED_RIDGE_ALPHA,
    "files": [
        {
            "file_name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }
        for path in files_to_package
    ],
    "validation_status": "passed",
}

temporary_manifest_path = (
    OUTPUT_MANIFEST_PATH.with_suffix(
        ".json.partial"
    )
)

with open(
    temporary_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        output_manifest,
        manifest_file,
        indent=2,
    )

temporary_manifest_path.replace(
    OUTPUT_MANIFEST_PATH
)

files_to_package.append(
    OUTPUT_MANIFEST_PATH
)

local_archive_path = (
    WORK_DIRECTORY
    / FINAL_OUTPUT_ARCHIVE_PATH.name
)

local_archive_path.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=1,
) as archive:
    for file_path in files_to_package:
        archive.write(
            file_path,
            arcname=file_path.name,
        )

with zipfile.ZipFile(
    local_archive_path,
    "r",
) as archive:
    damaged_member = archive.testzip()

    if damaged_member is not None:
        raise ValueError(
            f"The final archive contains a damaged file: {damaged_member}"
        )

    archived_members = set(
        archive.namelist()
    )

expected_members = {
    path.name
    for path in files_to_package
}

if archived_members != expected_members:
    raise ValueError(
        "The final archive member list is incomplete."
    )

local_archive_sha256 = file_sha256(
    local_archive_path
)

partial_archive_path = (
    FINAL_OUTPUT_ARCHIVE_PATH.with_suffix(
        ".zip.partial"
    )
)

partial_archive_path.unlink(
    missing_ok=True
)

shutil.copy2(
    local_archive_path,
    partial_archive_path,
)

if file_sha256(
    partial_archive_path
) != local_archive_sha256:
    raise IOError(
        "The copied final archive does not match the locally validated archive."
    )

partial_archive_path.replace(
    FINAL_OUTPUT_ARCHIVE_PATH
)

with zipfile.ZipFile(
    FINAL_OUTPUT_ARCHIVE_PATH,
    "r",
) as archive:
    if archive.testzip() is not None:
        raise ValueError(
            "The saved final archive failed validation."
        )

final_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Antibiotics",
            "value": EXPECTED_ANTIBIOTICS,
        },
        {
            "metric": "Completed matrix dimensions",
            "value": (
                f"{EXPECTED_MODEL_C_PATHOGENS} × "
                f"{EXPECTED_ANTIBIOTICS}"
            ),
        },
        {
            "metric": "Observed MIC cells preserved",
            "value": observed_count,
        },
        {
            "metric": "Predicted blank cells",
            "value": predicted_count,
        },
        {
            "metric": "Predicted cells with 95% prediction limits",
            "value": int(
                predicted_rows[
                    "lower_95_prediction_limit_log2_mic"
                ].notna().sum()
            ),
        },
        {
            "metric": "Unresolved cells",
            "value": unresolved_count,
        },
        {
            "metric": "Final archive size (MB)",
            "value": round(
                FINAL_OUTPUT_ARCHIVE_PATH.stat().st_size
                / (1024 ** 2),
                2,
            ),
        },
        {
            "metric": "Notebook 28 validation status",
            "value": "passed",
        },
    ]
)

display(final_summary)

print(
    f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}"
)

print(
    "\nNotebook 28 completed successfully. Observed MICs were preserved, "
    "blank cells were filled with Model C predictions, and every predicted "
    "MIC has antibiotic-specific 95% prediction limits and prediction-support "
    "information."
)

print(
    "Notebook 29 can next compare the completed Model 3, Model 3B and "
    "Model C matrices on their common BioSample cohort."
)
